# Module 11 – Curve Shocker

## Objective

This notebook demonstrates how to apply a parallel interest rate shock to a yield curve.

A Curve Shocker transforms an existing yield curve into a new shocked curve without modifying the original.

This component forms the foundation for:

- DV01
- Bucketed DV01
- PV01
- Scenario Analysis
- Stress Testing
- FRTB GIRR

Workflow

Original Curve
      │
      ▼
Curve Shocker
      │
      ▼
Shifted Curve

In [2]:
from datetime import date

from src.curves.bootstrapper import Bootstrapper
from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection
from src.risk.curve_shocker import CurveShocker

## Step 1

Construct the original yield curve from market quotes.

In [4]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.048))
quotes.add(MarketInstrument("Deposit", "3M", 0.050))
quotes.add(MarketInstrument("Deposit", "6M", 0.052))
quotes.add(MarketInstrument("Deposit", "1Y", 0.055))
quotes.add(MarketInstrument("Deposit", "2Y", 0.057))
quotes.add(MarketInstrument("Deposit", "3Y", 0.059))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve.summary()

Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8000%              0.996008
3M          0.2500        5.0000%              0.987578
6M          0.5000        5.2000%              0.974335
1Y          1.0000        5.5000%              0.946485
2Y          2.0000        5.7000%              0.892258
3Y          3.0000        5.9000%              0.837780
------------------------------------------------------------------------
Interpolation : Linear
Curve Points  : 6


## Step 2

Apply a parallel +1 basis point shift to the curve.

In [6]:
shocker = CurveShocker()

shifted_curve = shocker.parallel_shift(
    curve,
    0.0001,
)

## Step 3

Compare the original and shifted zero rates.

In [7]:
print(f"{'Tenor':<8} {'Original':>12} {'Shifted':>12}")

for original, shifted in zip(curve, shifted_curve):
    print(
        f"{original.tenor:<8}"
        f"{original.zero_rate:>12.6f}"
        f"{shifted.zero_rate:>12.6f}"
    )

Tenor        Original      Shifted
1M          0.048000    0.048100
3M          0.050000    0.050100
6M          0.052000    0.052100
1Y          0.055000    0.055100
2Y          0.057000    0.057100
3Y          0.059000    0.059100


## Step 4

Verify that the original curve remains unchanged.

In [8]:
print("Original Curve")
curve.summary()

print()

print("Shifted Curve")
shifted_curve.summary()

Original Curve
Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8000%              0.996008
3M          0.2500        5.0000%              0.987578
6M          0.5000        5.2000%              0.974335
1Y          1.0000        5.5000%              0.946485
2Y          2.0000        5.7000%              0.892258
3Y          3.0000        5.9000%              0.837780
------------------------------------------------------------------------
Interpolation : Linear
Curve Points  : 6

Shifted Curve
Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8100%              0.996000
3M          0.2500        5.0100%              0.987553
6M          0.5000        5.2100%              0.974286
1Y        

## Summary

In this module, we introduced the Curve Shocker, a reusable infrastructure component for transforming yield curves.

Key design principles:

- The original curve is immutable.
- Every shock returns a new YieldCurve instance.
- Pricing logic remains independent of risk transformations.

This architecture enables reusable curve transformations for:

- Parallel DV01
- Bucketed DV01
- PV01
- Scenario Analysis
- Historical VaR
- Stress Testing
- FRTB Standardised Approach